In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import sys, os
sys.path.append("/kaggle/input/datasets/nesmanasser/captionmodel")

!pip install -q rouge-score

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from src.data.dataset import load_captions, split_by_image, build_transforms, Flickr8kDataset
from src.data.preprocessing import Vocabulary
from src.models.caption_model import CaptionModel
from src.training.train import make_loader, train_model
from src.training.evaluate import evaluate_model

In [ ]:
IMAGES_DIR = "/kaggle/input/datasets/adityajn105/flickr8k/Images"
CAPTIONS_PATH = "/kaggle/input/datasets/adityajn105/flickr8k/captions.txt"

df = load_captions(CAPTIONS_PATH)
print(f"Total caption rows: {len(df):,}")
print(f"Unique images: {df['image'].nunique():,}")

SEED = 42
train_df, val_df, test_df = split_by_image(df, seed=SEED)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
vocab = Vocabulary(min_freq=5).build(train_df["caption"])
print(f"Vocabulary size: {len(vocab):,}")

MAX_LEN = 30
train_ds = Flickr8kDataset(train_df, IMAGES_DIR, vocab, MAX_LEN, build_transforms(train=True))
val_ds   = Flickr8kDataset(val_df,   IMAGES_DIR, vocab, MAX_LEN, build_transforms(train=False))
test_ds  = Flickr8kDataset(test_df,  IMAGES_DIR, vocab, MAX_LEN, build_transforms(train=False))

train_loader = make_loader(train_ds, batch_size=32, shuffle=True)
val_loader   = make_loader(val_ds,   batch_size=32, shuffle=False)

In [ ]:
model = CaptionModel(
    vocab_size=len(vocab),
    embed_dim=256,
    decoder_dim=512,
    attention_dim=512,
    fine_tune_from_block=6,
    dropout=0.5,
    pad_idx=vocab.pad_idx,
).to(device)

n_params = sum(p.numel() for p in model.trainable_parameters())
print(f"Trainable parameters: {n_params:,}")

In [ ]:
model, history = train_model(
    model, train_loader, val_loader, device,
    pad_idx=vocab.pad_idx,
    epochs=20, lr=3e-4, patience=4,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_name="best_model.pt",
)

In [ ]:
"""Greedy and beam-search caption generation for a single image at inference
time (no ground truth caption available — unlike training's teacher forcing)."""

import torch
from PIL import Image

from src.data.dataset import build_transforms
from src.data.preprocessing import Vocabulary
from src.models.caption_model import CaptionModel


@torch.no_grad()
def generate_caption(model: CaptionModel, image_path: str, vocab: Vocabulary,
                      device, max_len: int = 30, return_attention: bool = False):
    """Greedy decoding — picks the single best word at every step. Fast, but can
    get stuck in locally-good-but-globally-wrong choices (e.g. repeating a word)."""
    model.eval()

    transform = build_transforms(train=False)
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    encoder_out = model.encoder(image_tensor)
    num_pixels = encoder_out.size(1)

    h, c = model.decoder.init_hidden_state(encoder_out)
    token = torch.tensor([vocab.sos_idx], device=device)

    output_ids = []
    attn_steps = []

    for _ in range(max_len):
        embedding = model.decoder.embedding(token)
        context, alpha = model.decoder.attention(encoder_out, h)
        gate = model.decoder.sigmoid(model.decoder.f_beta(h))
        context = gate * context
        lstm_input = torch.cat([embedding, context], dim=1)
        h, c = model.decoder.lstm_cell(lstm_input, (h, c))
        logits = model.decoder.fc(h)
        next_id = int(logits.argmax(dim=1))
        attn_steps.append(alpha.squeeze(0).cpu())

        if next_id == vocab.eos_idx:
            break
        output_ids.append(next_id)
        token = torch.tensor([next_id], device=device)

    caption = vocab.decode(output_ids)

    if return_attention:
        attn_mat = torch.stack(attn_steps, dim=0) if attn_steps else torch.zeros(1, num_pixels)
        grid_size = int(num_pixels ** 0.5)
        return caption, attn_mat, grid_size

    return caption


@torch.no_grad()
def generate_caption_beam_search(model: CaptionModel, image_path: str,
                                  vocab: Vocabulary, device, beam_width: int = 5,
                                  max_len: int = 30):
    """Beam search decoding: instead of committing to the single best word at
    each step, keeps the top `beam_width` partial sequences alive at every
    step, and only commits at the end to whichever complete sequence has the
    highest total (length-normalized) log-probability. This avoids greedy
    decoding's failure mode of a locally-good choice leading to a bad overall
    sentence (e.g. getting stuck repeating a word).
    """
    model.eval()

    transform = build_transforms(train=False)
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)
    encoder_out = model.encoder(image_tensor)  # (1, num_pixels, encoder_dim)

    h0, c0 = model.decoder.init_hidden_state(encoder_out)

    # Each beam: (token_ids, log_prob, h, c). Start with beam_width copies of
    # just <start> so we can widen to beam_width unique paths after step 1.
    beams = [([vocab.sos_idx], 0.0, h0, c0)]
    completed = []

    for _ in range(max_len):
        candidates = []
        for token_ids, log_prob, h, c in beams:
            last_token = token_ids[-1]
            if last_token == vocab.eos_idx:
                # Already finished — carry it forward as-is, don't expand further.
                completed.append((token_ids, log_prob))
                continue

            token = torch.tensor([last_token], device=device)
            embedding = model.decoder.embedding(token)
            context, _ = model.decoder.attention(encoder_out, h)
            gate = model.decoder.sigmoid(model.decoder.f_beta(h))
            context = gate * context
            lstm_input = torch.cat([embedding, context], dim=1)
            new_h, new_c = model.decoder.lstm_cell(lstm_input, (h, c))
            logits = model.decoder.fc(new_h)  # (1, vocab_size)
            log_probs = torch.log_softmax(logits, dim=1).squeeze(0)  # (vocab_size,)

            top_log_probs, top_ids = log_probs.topk(beam_width)
            for lp, idx in zip(top_log_probs.tolist(), top_ids.tolist()):
                candidates.append((token_ids + [idx], log_prob + lp, new_h, new_c))

        if not candidates:
            break  # every beam already hit <end>

        # Keep only the best `beam_width` candidates overall (length-normalized,
        # so longer sequences aren't unfairly penalized for having more terms
        # summed into their log-probability).
        candidates.sort(key=lambda x: x[1] / len(x[0]), reverse=True)
        beams = candidates[:beam_width]

        if len(completed) >= beam_width:
            break

    completed.extend([(t, lp) for t, lp, _, _ in beams])
    best_ids, _ = max(completed, key=lambda x: x[1] / len(x[0]))

    # Strip <start>/<end> for decoding
    output_ids = [i for i in best_ids if i not in (vocab.sos_idx, vocab.eos_idx)]
    return vocab.decode(output_ids)

In [ ]:
metrics, examples_df = evaluate_model(model, test_df, IMAGES_DIR, vocab, device)
for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}" if isinstance(v, float) else f"{k:10s}: {v}")

examples_df.sample(10)

In [ ]:
import pickle
with open("/kaggle/working/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
print("Saved vocab.pkl alongside best_model.pt")